# jigsaw-bench: 2x2 puzzle with a small local VLM

Drives **Gemma 3 4B IT** (or **SmolVLM** as a fallback) through a 2x2 jigsaw puzzle using the cursor-style LLM interface.

**Kaggle setup:**
- Set the accelerator to a **T4** (or P100/L4) GPU.
- Turn **Internet** ON in the right-hand settings panel.
- For Gemma, accept the license on the model's HF page and add a Kaggle Secret named `HF_TOKEN`. If that's not set, the notebook auto-falls back to SmolVLM-Instruct (no auth needed).

Easy-mode is on: snap-to is position-only, so the VLM just has to drag each piece near its slot and release.

## 1. Install

In [ ]:
!pip install -q git+https://github.com/jerod92/jigsaw-bench.git
!pip install -q --upgrade transformers accelerate pillow

## 2. Build a small 2x2 puzzle

Small canvas keeps the rendered frame inside what a 4B-class VLM can still perceive at 640-768 px. Four high-contrast quadrants make pieces visually distinguishable.

In [ ]:
import numpy as np
from PIL import Image, ImageDraw

W, H = 240, 180
img = Image.new('RGB', (W, H), 'white')
draw = ImageDraw.Draw(img)
quad_colors = [(255, 80, 80), (80, 200, 80), (80, 130, 255), (240, 220, 60)]
for k, color in enumerate(quad_colors):
    qx = (k % 2) * (W // 2)
    qy = (k // 2) * (H // 2)
    draw.rectangle([qx, qy, qx + W // 2, qy + H // 2], fill=color)
draw.ellipse([W // 2 - 40, H // 2 - 40, W // 2 + 40, H // 2 + 40], fill='white', outline='black', width=3)
img

In [ ]:
from jigsaw_bench import generate_puzzle, shuffle_pieces, JigsawEnvironment

puzzle = generate_puzzle(img, width=W, height=H, n_cols=2, n_rows=2, seed=0)
layout = shuffle_pieces(
    puzzle,
    canvas_scale=2.0,
    rotation_deg_choices=(0, 90, 180, 270),  # quarter turns only — smaller worst-case bbox
    seed=0,
)
env = JigsawEnvironment(puzzle, layout)
frame = env.reset()
print('canvas:', frame.shape, '| pieces:', len(puzzle.pieces))
Image.fromarray(frame)

## 3. Load a small local VLM

Tries Gemma 3 4B IT first (gated, needs `HF_TOKEN`). Falls back to SmolVLM-Instruct (open) if that fails.

In [ ]:
import os, torch
from transformers import AutoProcessor

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if DEVICE == 'cuda' else torch.float32

model = None
processor = None
model_kind = None

try:
    from transformers import Gemma3ForConditionalGeneration
    GEMMA = 'google/gemma-3-4b-it'
    processor = AutoProcessor.from_pretrained(GEMMA)
    model = Gemma3ForConditionalGeneration.from_pretrained(
        GEMMA, torch_dtype=DTYPE, device_map=DEVICE,
    )
    model_kind = 'gemma3'
    print('Loaded Gemma 3 4B IT')
except Exception as e:
    print(f'Gemma load failed ({type(e).__name__}: {str(e)[:120]}...), falling back to SmolVLM')
    from transformers import AutoModelForVision2Seq
    SMOL = 'HuggingFaceTB/SmolVLM-Instruct'
    processor = AutoProcessor.from_pretrained(SMOL)
    model = AutoModelForVision2Seq.from_pretrained(SMOL, torch_dtype=DTYPE).to(DEVICE)
    model_kind = 'smolvlm'
    print('Loaded SmolVLM-Instruct')

## 4. Cursor-driving agent

We prompt the model to emit a single JSON tool call per step. The cursor is drawn on every frame so the model can see where it is.

In [ ]:
import json, re

SYSTEM_PROMPT = (
    "You are solving a 2x2 jigsaw puzzle. The image shows a working canvas with a dark gray "
    "rectangle (the target board) in the center, four scattered puzzle pieces around it, and "
    "a red circular cursor with a white crosshair. Your job: move the cursor onto a scattered "
    "piece, grab it, drag it to its correct location on the board, and release. Easy-mode "
    "snap is enabled, so releasing near the right slot is enough.\n\n"
    "Coordinate convention: dx>0 moves right, dy>0 moves down. The canvas is several hundred "
    "pixels across; typical move steps are 50-200 px.\n\n"
    "Respond with exactly ONE JSON object per step, nothing else. Tools:\n"
    "  {\"tool\": \"move\", \"dx\": <px>, \"dy\": <px>}\n"
    "  {\"tool\": \"grab\"}\n"
    "  {\"tool\": \"release\"}\n"
    "  {\"tool\": \"finished\"}\n"
)

TOOL_RE = re.compile(r'\{[^{}]*"tool"[^{}]*\}', re.DOTALL)

def parse_tool(text: str):
    m = TOOL_RE.search(text)
    if not m:
        return ('move', {'dx': 0, 'dy': 0})
    try:
        obj = json.loads(m.group(0))
    except Exception:
        return ('move', {'dx': 0, 'dy': 0})
    tool = obj.pop('tool', 'move')
    return (tool, obj)

@torch.inference_mode()
def vlm_call(frame_pil: Image.Image, step: int) -> str:
    user_text = f'Step {step}. Emit one JSON tool call.'
    if model_kind == 'gemma3':
        messages = [
            {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
            {'role': 'user', 'content': [
                {'type': 'image', 'image': frame_pil},
                {'type': 'text', 'text': user_text},
            ]},
        ]
        inputs = processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors='pt',
        ).to(model.device)
        in_len = inputs['input_ids'].shape[-1]
        out = model.generate(**inputs, max_new_tokens=80, do_sample=False)
        return processor.decode(out[0][in_len:], skip_special_tokens=True)
    # SmolVLM
    messages = [{'role': 'user', 'content': [
        {'type': 'image'},
        {'type': 'text', 'text': SYSTEM_PROMPT + '\n\n' + user_text},
    ]}]
    chat = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=chat, images=[frame_pil], return_tensors='pt').to(DEVICE)
    out = model.generate(**inputs, max_new_tokens=80, do_sample=False)
    return processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0]

def vlm_agent(frame: np.ndarray, step: int, tool_schema: dict):
    # Resize for the VLM — keeps aspect ratio, caps long side at 768.
    h, w = frame.shape[:2]
    scale = 768 / max(h, w)
    pil = Image.fromarray(frame).resize((int(w * scale), int(h * scale)))
    text = vlm_call(pil, step)
    if step <= 3:
        print(f'[step {step}] model said: {text!r}')
    return parse_tool(text)

## 5. Run the benchmark

In [ ]:
from jigsaw_bench import benchmark_llm

result = benchmark_llm(
    env, vlm_agent,
    max_steps=80,
    snap_to=True,
    snap_pos_threshold_px=80,         # generous for a small VLM
    snap_rot_threshold_deg=180,       # position-only snap
    pos_tol_px=12,
    rot_tol_deg=10,
)
result.summary()

## 6. Final canvas

In [ ]:
Image.fromarray(env.render())

## 7. Per-piece error breakdown

In [ ]:
for idx, err in sorted(result.final_position_error_px.items()):
    rot = result.final_rotation_error_deg[idx]
    ok = 'OK' if result.piece_correct[idx] else 'miss'
    print(f'piece {idx}: pos {err:6.1f} px | rot {rot:6.1f} deg | {ok}')